# Thessaloniki interactive map

An atlas of the city's urban infrastructure, organised around three
nested administrative zones — Urban, Metropolitan and Regional.

All the logic lives in the `thessmap` package. This notebook walks
through the map one theme at a time, previewing each, and saves the
finished map at the end. Nothing here needs editing to change how the
map looks — that lives in `src/thessmap/palette.py` and
`src/thessmap/registry.py`.

In [ ]:
from thessmap import build_map, config, palette, registry
from thessmap.data import MapData

data = MapData()

print("Project root:", config.PROJECT_ROOT)
print("Layers registered:", len(registry.LAYERS))

## How the previews work

`build_map(only=[...])` renders a subset.

Two arguments matter for previewing. `show_all=True` switches every
layer on — most default to off in the finished map, so a preview would
otherwise open empty. And `zoom` matters because detail is gated by zoom
level: bus stop symbols only exist from 16, trees from 18.

Each preview embeds its map HTML into this notebook, so run
**Clear All Outputs** before committing.

In [ ]:
# Central Thessaloniki, for previews that need street-level zoom
CENTRE = (40.6350, 22.9400)

# A close-up preview only needs data for what is on screen. Loading the
# whole region would embed megabytes of off-screen geometry per preview.
CLOSE_UP_BBOX = (22.91, 40.61, 22.98, 40.65)

close_up = MapData(bbox=CLOSE_UP_BBOX)

# Region-wide previews keep everything, but simplified: ~50 m is well
# below what is visible at zoom 9.
region = MapData(simplify=0.0005)


def preview(*layers, zoom=15, center=CENTRE, source=None):
    """Render a subset, everything switched on, at a chosen zoom."""
    return build_map(
        only=list(layers),
        data=source if source is not None else close_up,
        show_all=True,
        zoom=zoom,
        center=center,
        verbose=False,
    ).map

## 1. The layer registry

Every layer is declared once. A **parent** layer is what you toggle in
the menu; a **detail** layer switches on automatically inside a zoom
range. Nothing else in the codebase hard-codes a zoom threshold.

In [ ]:
import pandas as pd

pd.DataFrame([
    {
        "layer": spec.label,
        "parent": registry.spec(spec.parent).label,
        "from zoom": spec.min_zoom if spec.min_zoom is not None else "-",
        "to zoom": spec.max_zoom if spec.max_zoom is not None else "-",
    }
    for spec in registry.DETAILS
])

## 2. Zones — the study area

Three nested zones, dissolved into one study boundary that every other
layer is clipped to. This is what keeps 1.4 million building footprints
down to the 187,000 inside the region.

Hover any area for its name and zone.

In [ ]:
print(data.units["zone"].value_counts(), "\n")
print("Boundary area:", round(data.study_boundary.area.iloc[0] / 1e6), "km²")

preview("zones", zoom=9, center=None, source=region)

## 3. Water and lakes

Water is clipped to the study area at build time, since the source
covers far more than the region.

In [ ]:
print("Lakes:", len(region.selected_lakes))
print("Water polygons:", len(region.water_polygons))
print("Water lines:", len(region.water_lines))

preview("zones", "lakes", "water", zoom=9, center=None, source=region)

## 4. Metro and ferry

Metro stations sit at the top of the transport hierarchy. Zoom past 14
and the metro line changes style; past 15 and the stations gain their
service-intensity halo and a larger symbol.

In [ ]:
preview("zones", "metro_line", "metro_stations", "ferry_routes",
        "ferry_terminals", zoom=11, center=None, source=region)

## 5. Bus stops and service intensity

The clearest cartographic idea in the map: each stop's outer circle is
sized by how many lines serve it, so network hierarchy reads at a glance
rather than as a uniform scatter of dots.

Zoom 13 gives plain dots, 15 adds the service circles, 16 reveals the
station-type symbols.

In [ ]:
from thessmap.render.layers.bus import outer_radius

print(data.bus_stops["service_level"].value_counts(), "\n")
for line_count in [0, 1, 3, 6, 12]:
    print(f"  {line_count:>2} lines -> radius {outer_radius(line_count)}")

preview("bus_stops", "bus_lanes", zoom=15, center=CENTRE)

## 6. Bike network

Primary lanes show whenever the layer is on. Secondary lanes and the
clustered parking/rental points are street detail, held back to zoom 16.

In [ ]:
print("Primary lanes:", len(data.bike_lanes_primary))
print("Secondary lanes:", len(data.bike_lanes_secondary))
print("Parking:", len(data.bike_parking), " Rental:", len(data.bike_rental))

preview("bike", zoom=16, center=CENTRE)

## 7. Parking and taxi

Parking polygons appear at zoom 15, their P symbols at 17. Taxi ranks
at 17.

In [ ]:
preview("parking", "taxi", zoom=17, center=CENTRE)

## 8. Education and culture

Both arrive as a mix of polygons and points across several sources, so
preparation derives one symbol point per feature. Polygons from zoom 15,
symbols from 17.

In [ ]:
print(data.culture_symbols["culture_subtype"].value_counts(), "\n")

preview("education", "culture", zoom=17, center=CENTRE)

## 9. Trees

The densest layer — 42,000 points — so it is gated to zoom 18 alone.
Colour and size both encode the height class.

In [ ]:
print(data.trees["tree_class"].value_counts(), "\n")
for tree_class, (colour, radius) in palette.TREE_CLASSES.items():
    print(f"  {tree_class:<12} {colour}  radius {radius}")

preview("trees", zoom=18, center=CENTRE)

## 10. Everything together

The whole map over central Thessaloniki, buildings included. Cut to the
close-up window, so this stays a few megabytes rather than a hundred.

In [ ]:
preview(
    "zones", "water", "buildings", "bus_lanes", "bike", "bus_stops",
    "metro_line", "metro_stations", "parking", "taxi",
    "education", "culture", "trees",
    zoom=16,
)

## 11. Save the finished map

Every layer including buildings, written to `outputs/`.

Not displayed inline: with all layers the page is over 100 MB, because
Folium inlines every geometry into the HTML. Open the saved file in a
browser instead.

In [ ]:
builder = build_map(data=data)
output_path = builder.save()

print("\n" + "=" * 60)
print("SAVED:", output_path)
print("=" * 60)
print(f"  {output_path.stat().st_size / 1e6:.1f} MB")
print(f"  {len(builder.groups)} feature groups, "
      f"{len(builder.zoom_rules())} zoom rules")
print(f"\nOpen it with:\n  open {output_path}")

The same thing from the command line, without Jupyter:

```bash
python scripts/build_map.py
python scripts/build_map.py --only zones metro_line metro_stations
```